# AI Agent Demo

Minimal LangChain proof of concept that loads tools from the asyncroscopy MCP server and asks the model to use one or two of them. This will be useful to test our MCP server and new tools.

## Prereqs

Start the Tango stack on the microscope computer. Start MCP and the LLM device on the LLM computer. Both MCP and the LLM device use the microscope computer's Tango database:

```bash
uv run startup_scripts/run_servers.py
uv run startup_scripts/run_mcp.py --yaml configs/mcp.yaml
uv run startup_scripts/run_llm.py --yaml configs/gemma-llm.yaml
```

If not already done, install the optional AI extras:

```bash
uv sync --extra agent
# for local models do uv sync --extra agent --extra localagent
```

## Imports

In [1]:
import json
import os

import tango
from tiled.client import from_uri

## Ping Servers

In [2]:
MICROSCOPE_HOST = "localhost"
DB_PORT = 9094

os.environ["TANGO_HOST"] = f"{MICROSCOPE_HOST}:{DB_PORT}"
print(os.environ["TANGO_HOST"])

microscope = tango.DeviceProxy("asyncroscopy/instrument/default")
data = tango.DeviceProxy("asyncroscopy/data/default")
llm = tango.DeviceProxy("asyncroscopy/llm/default")

for proxy in (microscope, data, llm):
    proxy.set_timeout_millis(120_000)
    proxy.ping()
    print(proxy.name(), proxy.state())

tiled_config = json.loads(data.get_config())
client = from_uri(tiled_config["uri"])
print("Tiled:", tiled_config["uri"])

localhost:9094
asyncroscopy/instrument/default ON
asyncroscopy/data/default ON
asyncroscopy/llm/default ON
Tiled: http://localhost:9091


## Give a prompt

In [10]:
# adjust llm.max_steps if the LLM cannot complete a task
prompt = "Get a scanned image"
response = llm.query(prompt)
print(response)

The scanned image has been acquired and saved as `stem_image__20260727T110643543063.h5`.


In [4]:
import sys
import threading
import tango

def ask_llm(prompt: str):
    """Streams LLM query output directly using native Tango Change Events (zero polling lag)."""
    last_len = [0]
    done_event = threading.Event()

    def on_token_event(event: tango.EventData):
        if event.err or not event.attr_value:
            return
        
        current_text = event.attr_value.value or ""
        if len(current_text) > last_len[0]:
            sys.stdout.write(current_text[last_len[0]:])
            sys.stdout.flush()
            last_len[0] = len(current_text)

    # Subscribe directly to token updates
    event_id = llm.subscribe_event("token_stream", tango.EventType.CHANGE_EVENT, on_token_event)

    try:
        # Trigger query in background
        llm.Stream_Query(prompt)
        
        # Wait until state transitions back from RUNNING to ON
        while True:
            state = llm.State()
            if state != tango.DevState.RUNNING:
                break
            done_event.wait(0.02)

    finally:
        # Clean up event subscription
        llm.unsubscribe_event(event_id)

    print()  # Final newline

In [7]:
ask_llm("Get a scanned image")

Action: AutoScriptMicroscope_get_parameters
Arguments: {}

KeyboardInterrupt: 